In [3]:
import requests
import json
import time
import os
from datetime import datetime

API_URL = "https://www.kochwiki.org/api.php"
OUTPUT_DIR = "kochwiki_data"
DELAY = 0.5

In [4]:
def get_all_pages(namespace=0, limit=50):
    all_pages = []
    params = {
        "action": "query",
        "list": "allpages",
        "apnamespace": namespace,
        "aplimit": limit,
        "format": "json"
    }
    
    while True:
        print(f"  Fetching pages... (bisher: {len(all_pages)})")
        response = requests.get(API_URL, params=params)
        data = response.json()
        
        pages = data["query"]["allpages"]
        all_pages.extend(pages)
        
        if "continue" in data:
            params["apcontinue"] = data["continue"]["apcontinue"]
        else:
            break
        
        time.sleep(DELAY)
    
    print(f"  → {len(all_pages)} pages found in namespace {namespace}")
    return all_pages

def get_page_content(title):
    params = {
        "action": "query",
        "titles": title,
        "prop": "revisions|categories",
        "rvprop": "content|timestamp",
        "rvslots": "main",
        "cllimit": "50",
        "format": "json"
    }
    
    response = requests.get(API_URL, params=params)
    data = response.json()
    
    pages = data["query"]["pages"]
    page_id = list(pages.keys())[0]
    
    if page_id == "-1":
        return None
    
    page = pages[page_id]
    
    result = {
        "page_id": int(page_id),
        "title": page["title"],
        "wikitext": page["revisions"][0]["slots"]["main"]["*"],
        "timestamp": page["revisions"][0]["timestamp"],
        "categories": [cat["title"].replace("category:", "") 
                       for cat in page.get("categories", [])]
    }
    
    return result


def get_page_plaintext(title):
    params = {
        "action": "query",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,  
        "exsectionformat": "plain",
        "format": "json"
    }
    
    response = requests.get(API_URL, params=params)
    data = response.json()
    
    pages = data["query"]["pages"]
    page_id = list(pages.keys())[0]
    
    if page_id == "-1":
        return None
    
    return data["query"]["pages"][page_id].get("extract", "")


def get_categories():
    all_categories = []
    params = {
        "action": "query",
        "list": "allcategories",
        "aclimit": 500,
        "format": "json"
    }
    
    while True:
        response = requests.get(API_URL, params=params)
        data = response.json()
        
        categories = data["query"]["allcategories"]
        all_categories.extend([cat["*"] for cat in categories])
        
        if "continue" in data:
            params["accontinue"] = data["continue"]["accontinue"]
        else:
            break
        
        time.sleep(DELAY)
    
    print(f"→ {len(all_categories)} categories found")
    return all_categories


def get_pages_in_category(category, namespace=0):
    all_pages = []
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": f"Kategorie:{category}",
        "cmnamespace": namespace,
        "cmlimit": 500,
        "format": "json"
    }
    
    while True:
        response = requests.get(API_URL, params=params)
        data = response.json()
        
        members = data["query"]["categorymembers"]
        all_pages.extend(members)
        
        if "continue" in data:
            params["cmcontinue"] = data["continue"]["cmcontinue"]
        else:
            break
        
        time.sleep(DELAY)
    
    print(f"→ {len(all_pages)} pages in category '{category}'")
    return all_pages


def scrape_all(namespaces=None):
    if namespaces is None:
        namespaces = {
            0: "rezepte",         
            100: "zutaten",       
            102: "zubereitungen", 
        }
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    for ns_id, ns_name in namespaces.items():
        print(f"\n{'='*50}")
        print(f"Scraping Namespace: {ns_name} (ID: {ns_id})")
        print(f"{'='*50}")
        
        pages = get_all_pages(namespace=ns_id)
        
        all_content = []
        for i, page in enumerate(pages):
            title = page["title"]
            print(f"  [{i+1}/{len(pages)}] {title}")
            
            content = get_page_content(title)
            if content:
                all_content.append(content)
            
            time.sleep(DELAY)
        
        output_file = os.path.join(OUTPUT_DIR, f"{ns_name}.json")
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(all_content, f, ensure_ascii=False, indent=2)
        
        print(f"\n→ {len(all_content)} pages saved to {output_file}")
    
    print(f"\n{'='*50}")
    print("Scraping Categories...")
    print(f"{'='*50}")
    categories = get_categories()
    
    cat_file = os.path.join(OUTPUT_DIR, "kategorien.json")
    with open(cat_file, "w", encoding="utf-8") as f:
        json.dump(categories, f, ensure_ascii=False, indent=2)
    
    print(f"\nDone! All data saved in '{OUTPUT_DIR}/'")

In [5]:
print("=== KOCHWIKI SCRAPER - Quick Test ===\n")
    
    
print("1. Teste: Erste 5 Seiten holen...")
params = {
    "action": "query",
    "list": "allpages",
    "apnamespace": 0,
    "aplimit": 5,
    "format": "json"
    }
resp = requests.get(API_URL, params=params)
pages = resp.json()["query"]["allpages"]
for p in pages:
    print(f"   - {p['title']}")
    

print(f"\n2. Teste: Inhalt von '{pages[0]['title']}' holen...")
content = get_page_content(pages[0]["title"])
if content:
    print(f"   Titel: {content['title']}")
    print(f"   Kategorien: {content['categories']}")
    print(f"   Wikitext (erste 300 Zeichen):")
    print(f"   {content['wikitext'][:300]}...")
    

print(f"\n3. Teste: Seiten in 'Deutsche Küche'...")
deutsche = get_pages_in_category("Deutsche Küche")
for p in deutsche[:5]:
    print(f"   - {p['title']}")
    
print("\n" + "="*50)
print("Quick Test abgeschlossen!")
print("Wenn alles funktioniert, starte den vollen Scrape mit:")
print("  scrape_all()")
print("="*50)

=== KOCHWIKI SCRAPER - Quick Test ===

1. Teste: Erste 5 Seiten holen...
   - "Krömpele"-Suppe
   - 123 Teig
   - 19th Green
   - 1 2 3 Teig
   - 2000 Miles away

2. Teste: Inhalt von '"Krömpele"-Suppe' holen...
   Titel: "Krömpele"-Suppe
   Kategorien: ['Kategorie:Brotsuppen', 'Kategorie:Nocken', 'Kategorie:Rezepte ohne Alkohol', 'Kategorie:Schwierigkeit leicht', 'Kategorie:Thüringer Küche']
   Wikitext (erste 300 Zeichen):
   {{Rezept|
 | Menge         = 4 Personen
 | Zeit          = 30–40 Minuten
 | Schwierigkeit = leicht
 | Alkohol       = nein
 | Bild          = Kein_Bild.png
|}}

Die '''"Krömpele"-Suppe''' ist eine typische thüringische Suppe. ''Krömpele'' ist mundartlich und steht für ''Krümel'', ''Streusel'', ''Brö...

3. Teste: Seiten in 'Deutsche Küche'...
→ 472 pages in category 'Deutsche Küche'
   - Abgebackene Klöße
   - Acerola-Apfelgelee
   - Ananasgelee
   - Angemachter Camembert
   - Angemachter Weichkäse

Quick Test abgeschlossen!
Wenn alles funktioniert, starte den v

In [5]:
for zutat in ["Zutat:Mehl", "Zutat:Salz", "Zutat:Butter", "Zutat:Ei"]:
    content = get_page_content(zutat)
    if content:
        print(f"=== {zutat} ===")
        print(content["wikitext"][:1000])
        print(f"\nKategorien: {content['categories']}")
        print("\n---END---\n")

=== Zutat:Mehl ===
{{Zutatübersicht|
 | Bild          = Flours.jpg
|}}
'''Mehl''' ist gemahlenes [[Zutat:Getreide|Getreide]] und wird als Basis von Teigen zur Herstellung von Backwaren oder Teigwaren verwendet. Wird in in Rezepten nur von „Mehl“ gesprochen, ist für gewöhnlich [[Zutat:Weizenmehl|Weizenmehl]] der Type 405, bei Broten Type 550, gemeint.

== Typisierung nach DIN ==
Die Ermittlung der Type bzw. Helligkeit erfolgt durch die Bestimmung des Mineralstoffgehaltes. Niedrige Mehl-Typen wie 405 sind – mit geringem Mineralstoffgehalt – sehr hell, hohe Typen wie 1800 sehr dunkel und reich an Mineralstoffen. Die Typenzahl bezeichnet hierbei den Mineralstoffgehalt in Gramm pro 100 Kilogramm wasserfreiem Mehl. Unter Laborbedingungen wird hierzu  eine geringe Menge des Mehls bei {{Grad|900}} im Muffelofen verbrannt. Die verbleibenden (nichtbrennbaren Bestandteile) entsprechen im Wesentlichen der Mineralstoffmenge des Mehles. Fälschlicherweise werden diese Bestandteile deshalb auch umgang